In [1]:
import os
import ollama
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

/Users/samarthmn/Projects/samarthmn/data-alchemy/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Available ollama models locally


def _format_size(size_bytes: int) -> str:
    size = float(size_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024


OllamaHost = os.getenv("OLLAMA_HOST")
OllamaBaseUrl = f"{OllamaHost}/v1"
OllamaClient = ollama.Client(host=OllamaHost)

available_models = [
    {
        "model": m.model,
        "size": _format_size(m.size or 0),
        "params": m.details.parameter_size if m.details else "—",
        "quant": m.details.quantization_level if m.details else "—",
        "modified": m.modified_at.strftime("%Y-%m-%d %H:%M") if m.modified_at else "—",
    }
    for m in OllamaClient.list().models
]

rows = "\n".join(f"{m['model']}" for m in available_models)
print(rows)

qwen3.6:35b
gemma4:31b
gpt-oss:20b
gemma4:26b
nomic-embed-text:latest
gemma4:e4b


In [3]:
MODEL = "gpt-oss:20b"

In [4]:
system_message = """
You are DataAlchemy, a synthetic data generator that creates realistic, internally consistent datasets from a user's idea.

Your job is to generate the requested dataset from already-collected requirements and return a concise description of what was generated.

Required inputs:
1. topic: the dataset subject, domain, or idea.
2. complexity: one of Simple, Medium, or Hard.
   - Simple: fewer fields, fewer rows, minimal variation, and little or no linking between entities.
   - Medium: more fields and rows, moderate variation, and some relationships between entities.
   - Hard: many fields and rows, high variation, multiple linked entities, and realistic edge cases.
3. format: one of CSV, JSON, or Markdown.

When generating data, make it realistic for the selected topic. Use plausible field names, values, IDs, dates, categories, and relationships. Avoid placeholder values such as "foo", "test", "sample", or repeated generic rows unless the topic requires them.

Create table-shaped datasets, not a single flattened blob. Simple datasets may use one table. Medium datasets must include at least two related tables. Hard datasets must include at least three related tables with realistic primary-key and foreign-key columns.

Return one file per table in the requested format. For CSV, return files such as customers.csv and orders.csv. For JSON, return files such as customers.json and orders.json, each containing an array of row objects for that table; do not put the whole dataset in one JSON file. For Markdown, return one table per .md file.

Return real dataset files for the application runtime to package into a zip. Do not return a fake zip reference and do not claim that a file was created outside this response.

Always return exactly one valid JSON object. Do not include Markdown fences, comments, or text outside the JSON.
The status value must be exactly one of: "success" or "error".

Response schema:
{
    "status": "success",
    "reply": "Generated the dataset.",
    "data": {
        "description": "Dataset description.",
        "files": [
            {
                "filename": "customers.csv",
                "content": "table content"
            },
            {
                "filename": "orders.csv",
                "content": "related table content"
            }
        ]
    },
    "error": {
        "message": "A clear explanation of what went wrong and how the user can retry."
    }
}

Include only fields that apply to the current status: data for "success" and error for "error".
"""

checker_message = """
You extract requirements for a synthetic dataset request.

Return exactly one valid JSON object with this schema:
{
    "topic": "dataset subject or null",
    "complexity": "Simple, Medium, Hard, or null",
    "format": "CSV, JSON, Markdown, or null",
    "missing": ["topic", "complexity", "format"],
    "reply": "short user-facing reply"
}

Rules:
- Use the provided current state unless the latest user message overrides it.
- Extract complexity only if the user explicitly wrote Simple, Medium, or Hard. Do not infer it from the topic.
- Extract format only if the user explicitly wrote CSV, JSON, or Markdown. Do not infer it from the topic.
- If there is no clear dataset subject, set topic to null.
- Include every still-missing field in missing.
- If topic is missing, ask the user to type the dataset topic.
- If only complexity or format is missing, keep reply short because the app will show option cards.
- Do not generate data.
"""

In [5]:
import json
import re
import tempfile
import threading
import zipfile
from pathlib import Path

openai = OpenAI(base_url=OllamaBaseUrl, api_key="ollama")

REQUIRED_FIELDS = ("topic", "complexity", "format")
COMPLEXITY_OPTIONS = ("Simple", "Medium", "Hard")
FORMAT_OPTIONS = ("CSV", "JSON", "Markdown")


def explicit_complexity(message):
    match = re.search(r"\b(simple|medium|hard)\b", message, re.IGNORECASE)
    return match.group(1).title() if match else None


def explicit_format(message):
    match = re.search(r"\b(csv|json|markdown|md)\b", message, re.IGNORECASE)
    if not match:
        return None
    value = match.group(1).lower()
    return "Markdown" if value in {"markdown", "md"} else value.upper()


def required_file_count(complexity):
    return {"Simple": 1, "Medium": 2, "Hard": 3}.get(complexity, 1)


def split_single_json_dataset(files):
    if len(files or []) != 1:
        return files
    filename = str(files[0].get("filename") or "dataset.json").lower()
    if not filename.endswith(".json"):
        return files
    try:
        dataset = json.loads(files[0].get("content") or "")
    except (TypeError, json.JSONDecodeError):
        return files
    if not isinstance(dataset, dict):
        return files

    split_files = []
    for table_name, rows in dataset.items():
        if not isinstance(rows, (list, dict)):
            continue
        safe_name = (
            re.sub(r"[^a-zA-Z0-9_-]+", "_", str(table_name)).strip("_") or "table"
        )
        split_files.append(
            {
                "filename": f"{safe_name}.json",
                "content": json.dumps(rows, indent=2),
            }
        )
    return split_files or files


def call_llm_json(messages):
    result = (
        openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            response_format={"type": "json_object"},
            max_tokens=8192,
        )
        .choices[0]
        .message.content
    )
    text = result.strip()
    if text.startswith("```"):
        text = text.strip("`").strip()
        if text.lower().startswith("json"):
            text = text[4:].strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise
        return json.loads(text[start : end + 1])


def make_zip(files):
    output_dir = Path(tempfile.mkdtemp(prefix="dataalchemy_"))
    zip_path = output_dir / "dataalchemy_dataset.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for index, file_info in enumerate(files or [], start=1):
            filename = Path(
                str(file_info.get("filename") or f"dataset_{index}.txt")
            ).name
            content = file_info.get("content", "")
            archive.writestr(filename, str(content))
    return str(zip_path)


def next_question_card(state):
    """Return the option card for the next missing field, or None if complete."""
    if not state.get("complexity"):
        return gr.ChatMessage(
            role="assistant",
            content="How complex should the dataset be?",
            options=[
                {"label": option, "value": f"complexity:{option}"}
                for option in COMPLEXITY_OPTIONS
            ],
        )
    if not state.get("format"):
        return gr.ChatMessage(
            role="assistant",
            content="Which format would you like?",
            options=[
                {"label": option, "value": f"format:{option}"}
                for option in FORMAT_OPTIONS
            ],
        )
    return None


def _spinner_message(caption):
    """A 'thinking' bubble with a live spinner and a single static caption."""
    return gr.ChatMessage(
        role="assistant",
        content="",
        metadata={"title": caption, "status": "pending"},
    )


def stream_thinking(chat, holder, work_fn, caption):
    """Run work_fn() in a background thread while a thinking bubble spins.

    Shows a single spinner (it animates client-side) until the work finishes.
    The outcome is stored in holder as {"result": ...} or {"error": ...}.
    """
    holder.clear()

    def runner():
        try:
            holder["result"] = work_fn()
        except Exception as exc:  # surfaced to the caller via holder
            holder["error"] = exc

    thread = threading.Thread(target=runner, daemon=True)
    thread.start()

    chat.append(_spinner_message(caption))
    yield chat
    thread.join()
    chat.pop()


def build_generation_message(state):
    """Call the generator model and return a single assistant ChatMessage."""
    messages = [
        {"role": "system", "content": system_message},
        {
            "role": "user",
            "content": json.dumps(
                {
                    "topic": state["topic"],
                    "complexity": state["complexity"],
                    "format": state["format"],
                    "minimum_files": required_file_count(state["complexity"]),
                    "file_rule": "Return one file per table. Medium needs at least two related tables. Hard needs at least three related tables.",
                }
            ),
        },
    ]
    try:
        result = call_llm_json(messages)
    except Exception as exc:
        return gr.ChatMessage(
            role="assistant",
            content=f"I couldn't parse the generator response. Try again. ({exc})",
        )

    if result.get("status") != "success":
        message = (
            result.get("error", {}).get("message")
            or result.get("reply")
            or "Generation failed. Try again."
        )
        return gr.ChatMessage(role="assistant", content=message)

    data = result.get("data") or {}
    files = split_single_json_dataset(data.get("files") or [])
    if not files:
        return gr.ChatMessage(
            role="assistant",
            content="The generator did not return any files. Try again.",
        )
    minimum_files = required_file_count(state.get("complexity"))
    if len(files) < minimum_files:
        return gr.ChatMessage(
            role="assistant",
            content=(
                f"The generator returned {len(files)} file(s), but {state.get('complexity')} "
                f"datasets require at least {minimum_files} related table files. Please try again."
            ),
        )

    zip_path = make_zip(files)
    reply = result.get("reply") or "Generated the dataset."
    description = data.get("description") or "Dataset files are ready."
    return gr.ChatMessage(
        role="assistant",
        content=[
            f"{reply}\n\n{description}",
            gr.File(value=zip_path, label="Download dataset"),
        ],
    )


def stream_generation(chat, state):
    """Show a 'generating' spinner, then append the generated result."""
    holder = {}
    for snapshot in stream_thinking(
        chat,
        holder,
        lambda: build_generation_message(state),
        "Generating your dataset…",
    ):
        yield snapshot
    if "error" in holder:
        chat.append(
            gr.ChatMessage(
                role="assistant",
                content=f"I couldn't generate the dataset. Try again. ({holder['error']})",
            )
        )
    else:
        chat.append(holder["result"])
    yield chat


def clear_card_options(chat, field):
    """Remove the clickable buttons from an already-answered question card."""
    prefix = f"{field}:"
    for msg in chat:
        options = (
            msg.get("options")
            if isinstance(msg, dict)
            else getattr(msg, "options", None)
        )
        if not options:
            continue
        values = [
            (opt.get("value") if isinstance(opt, dict) else getattr(opt, "value", ""))
            or ""
            for opt in options
        ]
        if any(str(value).startswith(prefix) for value in values):
            if isinstance(msg, dict):
                msg["options"] = None
            else:
                msg.options = None


def handle_user_message(message, chat, state):
    chat = list(chat or [])
    state = {**{field: None for field in REQUIRED_FIELDS}, **(state or {})}
    message = (message or "").strip()
    if not message:
        yield chat, "", state
        return

    # Echo the user message and clear the textbox immediately.
    chat.append(gr.ChatMessage(role="user", content=message))
    yield chat, "", state

    # Initial loading: show a thinking spinner while the checker runs.
    checker_payload = {"current_state": state, "user_message": message}
    holder = {}
    for snapshot in stream_thinking(
        chat,
        holder,
        lambda: call_llm_json(
            [
                {"role": "system", "content": checker_message},
                {"role": "user", "content": json.dumps(checker_payload)},
            ]
        ),
        "Thinking…",
    ):
        yield snapshot, "", state
    if "error" in holder:
        chat.append(
            gr.ChatMessage(
                role="assistant",
                content=f"I couldn't understand that request. Please include a topic, complexity, and format. ({holder['error']})",
            )
        )
        yield chat, "", state
        return
    result = holder["result"]

    topic = str(result.get("topic") or "").strip()
    if topic and topic.lower() not in {"null", "none"}:
        state["topic"] = topic
    complexity = explicit_complexity(message)
    if complexity:
        state["complexity"] = complexity
    dataset_format = explicit_format(message)
    if dataset_format:
        state["format"] = dataset_format

    if not state.get("topic"):
        reply = result.get("reply") or "What dataset topic should I generate?"
        chat.append(gr.ChatMessage(role="assistant", content=reply))
        yield chat, "", state
        return

    # Ask one question at a time, step by step.
    card = next_question_card(state)
    if card:
        chat.append(card)
        yield chat, "", state
        return

    # Everything collected in one go -> generate with a live indicator.
    for snapshot in stream_generation(chat, state):
        yield snapshot, "", state


def handle_card_choice(chat, state, evt: gr.SelectData):
    chat = list(chat or [])
    state = {**{field: None for field in REQUIRED_FIELDS}, **(state or {})}
    choice = str(evt.value or "")
    if ":" not in choice:
        yield chat, state
        return

    field, value = choice.split(":", 1)
    if field == "complexity" and value in COMPLEXITY_OPTIONS:
        state["complexity"] = value
    elif field == "format" and value in FORMAT_OPTIONS:
        state["format"] = value
    else:
        yield chat, state
        return

    # Retire the answered card's buttons and echo the choice as a user message.
    clear_card_options(chat, field)
    chat.append(gr.ChatMessage(role="user", content=value))
    yield chat, state

    # Advance to the next missing question, if any.
    card = next_question_card(state)
    if card:
        chat.append(card)
        yield chat, state
        return

    if all(state.get(f) for f in REQUIRED_FIELDS):
        for snapshot in stream_generation(chat, state):
            yield snapshot, state


with gr.Blocks(title="DataAlchemy") as demo:
    gr.Markdown("# DataAlchemy")
    gr.Markdown("Generate a dataset based on the topic, complexity and format.")
    chatbot = gr.Chatbot(
        allow_file_downloads=True,
        height=520,
    )
    collected_state = gr.State({field: None for field in REQUIRED_FIELDS})

    with gr.Row():
        user_input = gr.Textbox(
            placeholder="Describe the dataset you want, e.g. retail orders, hard, CSV",
            show_label=False,
            scale=8,
        )
        send_button = gr.Button("Send", variant="primary", scale=1)
        clear_button = gr.Button("Clear", scale=1)

    send_button.click(
        handle_user_message,
        inputs=[user_input, chatbot, collected_state],
        outputs=[chatbot, user_input, collected_state],
    )
    user_input.submit(
        handle_user_message,
        inputs=[user_input, chatbot, collected_state],
        outputs=[chatbot, user_input, collected_state],
    )
    chatbot.option_select(
        handle_card_choice,
        inputs=[chatbot, collected_state],
        outputs=[chatbot, collected_state],
    )
    clear_button.click(
        lambda: ([], "", {field: None for field in REQUIRED_FIELDS}),
        outputs=[chatbot, user_input, collected_state],
    )


demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/Users/samarthmn/Projects/samarthmn/data-alchemy/.venv/lib/python3.14/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/samarthmn/Projects/samarthmn/data-alchemy/.venv/lib/python3.14/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/samarthmn/Projects/samarthmn/data-alchemy/.venv/lib/python3.14/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
